# Validation Step 2.5 — OpenBTAI Embedding Evaluation (Full Battery)

**Mirrors:** `Phase3_A4B_HybridBSF_Eval.ipynb` — same 16-test protocol adapted for OpenBTAI.

**What this does:**
1. Loads the 253 clean BSF hybrid embeddings (valid radiomics coverage only)
2. Computes raw targets from the actual NIfTI masks on disk (volume, elongation, subregions, GLCM)
3. Runs the same M/H/T battery as Cyprus using identical PCA→Ridge probes
4. Reports Pure BSF (9216-dim) vs previous Cyprus scores for comparison

**Domain gap note:** OpenBTAI uses single-modality T1c replicated 4x. The Cyprus model was trained on 4 distinct contrasts, so some texture/intensity tests will inherently score lower. Macro-geometry tests (volume, elongation) should transfer well.

In [11]:
import os, warnings, numpy as np, pandas as pd, nibabel as nib
from pathlib import Path
from scipy.stats import pearsonr, skew, kurtosis
from scipy.ndimage import label as nd_label
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
EMB_PATH        = Path("/home/moamed/canada_me/explainable_diseas/implementation_cyprus/Validation/openbtai_hybrid_embeddings_v2.npz")
PREPROCESS_ROOT = Path("/home/moamed/HDD/validation_data/preprocessed_openbtai")
TIMELINES_CSV   = PREPROCESS_ROOT / "openbtai_patient_timelines.csv"

print(f"Embeddings  : {EMB_PATH.exists()}  {EMB_PATH}")
print(f"Preprocessed: {PREPROCESS_ROOT.exists()}  {PREPROCESS_ROOT}")
print(f"Timelines   : {TIMELINES_CSV.exists()}")

Embeddings  : True  /home/moamed/canada_me/explainable_diseas/implementation_cyprus/Validation/openbtai_hybrid_embeddings_v2.npz
Preprocessed: True  /home/moamed/HDD/validation_data/preprocessed_openbtai
Timelines   : True


In [12]:
# ── Load embeddings ─────────────────────────────────────────────────────────
d = np.load(str(EMB_PATH), allow_pickle=True)
hybrid_embeddings = {k: d[k] for k in d.files}
bsf_embeddings    = {k: v[:9216] for k, v in hybrid_embeddings.items()}

print(f"Loaded {len(hybrid_embeddings)} hybrid embeddings")
print(f"  Hybrid dim : {list(hybrid_embeddings.values())[0].shape[0]}")
print(f"  Pure BSF   : 9216-dim (octant x8 + mask x4 = 9216)")

timelines = pd.read_csv(TIMELINES_CSV)
print(f"Timelines  : {len(timelines)} scans from {timelines['patient_id'].nunique()} patients")

Loaded 253 hybrid embeddings
  Hybrid dim : 9234
  Pure BSF   : 9216-dim (octant x8 + mask x4 = 9216)
Timelines  : 373 scans from 75 patients


In [13]:
# ── Probe infrastructure (identical to Phase3_A4B) ─────────────────────────
class _SafePCA(PCA):
    def fit_transform(self, X, y=None):
        self.n_components = min(self.n_components, X.shape[0]-1, X.shape[1])
        return super().fit_transform(X, y)
    def fit(self, X, y=None):
        self.n_components = min(self.n_components, X.shape[0]-1, X.shape[1])
        return super().fit(X, y)

def make_probe(task='regression', n_samples=None, n_dim=None):
    _n = n_samples or 253
    n_comp = min(85, max(2, _n // 2)) if (n_dim or 0) > 1000 else min(30, max(2, _n - 2))
    alpha = 10.0 if (n_dim or 0) > 1000 else 1.0
    if task == 'regression':
        return make_pipeline(StandardScaler(), _SafePCA(n_components=n_comp), Ridge(alpha=alpha))
    return make_pipeline(StandardScaler(), _SafePCA(n_components=n_comp),
                         LogisticRegression(max_iter=2000, C=0.005 if (n_dim or 0) > 1000 else 0.01))

CV_SCAN = KFold(n_splits=5, shuffle=True, random_state=42)
print("Probe pipeline ready.")
print(f"CV strategy: {CV_SCAN}")

Probe pipeline ready.
CV strategy: KFold(n_splits=5, random_state=42, shuffle=True)


In [14]:
# ── Compute target variables from raw NIfTI masks ─────────────────────────
# Same logic as Phase3_A4B — directly from the segmentation masks on disk
# This gives HONEST targets independent from the embedding shape appendage

volumes_dict = {}

for k in bsf_embeddings.keys():
    pid, visit = k.split('__')
    msk_path = PREPROCESS_ROOT / pid / visit / 'mask_subregions.nii.gz'
    if not msk_path.exists():
        msk_path = PREPROCESS_ROOT / pid / visit / 'mask_subregions.nii'
    if not msk_path.exists():
        continue
    try:
        img    = nib.load(str(msk_path))
        mask   = img.get_fdata()
        spacing = img.header.get_zooms()[:3]
        vvol   = float(np.prod(spacing))
        binary = (mask > 0).astype(np.uint8)
        wt_vol = float(binary.sum() * vvol)

        # Shape features
        coords = np.argwhere(binary)
        elong = 0.0; flatness = 0.0; sphericity = 0.0; svr = 0.0
        if len(coords) >= 4:
            cen = coords - coords.mean(0)
            cov = np.cov(cen.T)
            ev  = np.sort(np.abs(np.linalg.eigvalsh(cov)))  # [min, mid, max]
            elong    = float(np.sqrt(ev[0] / (ev[2] + 1e-8)))
            flatness = float(np.sqrt(ev[0] / (ev[1] + 1e-8)))
            surf     = float(4 * np.pi * (3 * wt_vol / (4 * np.pi)) ** (2/3))
            svr      = surf / (wt_vol + 1e-8)
            sphericity = float((np.pi**(1/3) * (6*wt_vol)**(2/3)) / (surf + 1e-8))

        label_vals      = mask[binary.astype(bool)]
        labels_present  = set(mask[mask > 0].astype(int).tolist())
        ncr_present     = int(1 in labels_present)
        et_present      = int(3 in labels_present)
        n_subregions    = len(labels_present)
        glcm_entropy    = float(np.log1p(np.var(label_vals))) if len(label_vals) > 0 else 0.0

        volumes_dict[k] = {
            'wt_volume_mm3': wt_vol,
            'elongation':    elong,
            'flatness':      flatness,
            'sphericity':    sphericity,
            'svr':           svr,
            'ncr_present':   ncr_present,
            'et_present':    et_present,
            'n_subregions':  n_subregions,
            'glcm_entropy':  glcm_entropy,
        }
    except Exception as e:
        pass

print(f"Extracted targets for {len(volumes_dict)} / {len(bsf_embeddings)} scans")
eval_keys = sorted([k for k in bsf_embeddings if k in volumes_dict])

X       = np.array([hybrid_embeddings[k] for k in eval_keys])
X_pure  = np.array([bsf_embeddings[k]    for k in eval_keys])
vol_arr     = np.array([volumes_dict[k]['wt_volume_mm3'] for k in eval_keys])
elong_arr   = np.array([volumes_dict[k]['elongation']    for k in eval_keys])
flat_arr    = np.array([volumes_dict[k]['flatness']      for k in eval_keys])
sph_arr     = np.array([volumes_dict[k]['sphericity']    for k in eval_keys])
svr_arr     = np.array([volumes_dict[k]['svr']           for k in eval_keys])
ncr_arr     = np.array([volumes_dict[k]['ncr_present']   for k in eval_keys])
sub_arr     = np.array([volumes_dict[k]['n_subregions']  for k in eval_keys])
glcm_arr    = np.array([volumes_dict[k]['glcm_entropy']  for k in eval_keys])
pat_arr     = np.array([k.split('__')[0] for k in eval_keys])

print(f"\nEval matrix — Hybrid: {X.shape}, Pure BSF: {X_pure.shape}")
print(f"Vol:  min={vol_arr.min():.0f}  max={vol_arr.max():.0f}  zeros={(vol_arr==0).sum()}")
print(f"Elong: min={elong_arr.min():.3f}  max={elong_arr.max():.3f}")

Extracted targets for 253 / 253 scans

Eval matrix — Hybrid: (253, 9234), Pure BSF: (253, 9216)
Vol:  min=11  max=29513  zeros=0
Elong: min=0.317  max=0.920


In [15]:
# ── MORPHOLOGY TESTS M1-M6 ─────────────────────────────────────────────────
# Cyprus A4B scores for comparison:
# M1 Vol    Hybrid=0.611  Pure=0.437
# M2 LogVol Hybrid=0.605  Pure=0.447
# M3 SVR    Hybrid=0.997  Pure=0.210
# M4 NCR    Hybrid=0.716  Pure=0.695
# M5 Elong  Hybrid=0.998  Pure=-0.072
# M6 NN     Hybrid=23.8%  Pure=23.8%

RESULTS_B = {}  # OpenBTAI BSF-only (pure BSF no shape)
CY_REF = {'M1':0.437,'M2':0.447,'M3':0.210,'M4':0.695,'M5':-0.072,'M6':23.8,
           'H1':0.189,'H2':-0.497,'H3':0.619,'H4':0.285,
           'T1':0.578,'T3':-0.549,'T4':0.509,'T5':0.942,'T6':0.563}

print("=" * 60)
print("  MORPHOLOGY TESTS M1-M6")
print("  (Cyprus Pure BSF scores shown for reference)")
print("=" * 60)

# M1: Volume
s = cross_val_score(make_probe(n_dim=X_pure.shape[1]), X_pure, np.log1p(vol_arr), cv=CV_SCAN, scoring='r2')
RESULTS_B['M1_volume_R2'] = s.mean()
status = "✅" if s.mean() > CY_REF['M1']*0.7 else "⚠️"
print(f"  {status} M1 Volume R²      OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['M1']:.3f}")

# M2: Log10 Volume
s = cross_val_score(make_probe(n_dim=X_pure.shape[1]), X_pure, np.log10(vol_arr+1), cv=CV_SCAN, scoring='r2')
RESULTS_B['M2_logvol_R2'] = s.mean()
status = "✅" if s.mean() > CY_REF['M2']*0.7 else "⚠️"
print(f"  {status} M2 LogVol R²      OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['M2']:.3f}")

# M3: SVR
valid_svr = svr_arr > 0
s = cross_val_score(make_probe(n_dim=X_pure.shape[1]), X_pure[valid_svr], svr_arr[valid_svr], cv=CV_SCAN, scoring='r2')
RESULTS_B['M3_svr_R2'] = s.mean()
status = "✅" if s.mean() > CY_REF['M3']*0.7 else "⚠️"
print(f"  {status} M3 SVR R²         OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['M3']:.3f}")

# M4: Necrosis classification
if len(np.unique(ncr_arr)) >= 2:
    s = cross_val_score(make_probe('classification', n_dim=X_pure.shape[1]), X_pure, ncr_arr, cv=CV_SCAN, scoring='f1')
    RESULTS_B['M4_necrosis_F1'] = s.mean()
    status = "✅" if s.mean() > CY_REF['M4']*0.7 else "⚠️"
    print(f"  {status} M4 Necrosis F1    OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['M4']:.3f}")
else:
    print("  ⚪ M4 Necrosis — insufficient class variation")

# M5: Elongation
valid_e = elong_arr > 0
if valid_e.sum() >= 20:
    s = cross_val_score(make_probe(n_dim=X_pure.shape[1]), X_pure[valid_e], elong_arr[valid_e], cv=CV_SCAN, scoring='r2')
    RESULTS_B['M5_elongation_R2'] = s.mean()
    status = "✅" if s.mean() > 0.10 else "⚠️"
    print(f"  {status} M5 Elongation R²  OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['M5']:.3f}")

# M6: NN volume consistency
X_sc = StandardScaler().fit_transform(X_pure)
nn   = NearestNeighbors(n_neighbors=6, metric='cosine').fit(X_sc)
_, idx = nn.kneighbors(X_sc)
consistent = total = 0
for i in range(len(X_pure)):
    if vol_arr[i] == 0: continue
    for j in idx[i, 1:]:
        if vol_arr[j] == 0: continue
        ratio = min(vol_arr[i], vol_arr[j]) / max(vol_arr[i], vol_arr[j])
        if ratio > 0.7: consistent += 1
        total += 1
RESULTS_B['M6_nn_pct'] = consistent/total*100 if total else 0
status = "✅" if RESULTS_B['M6_nn_pct'] > CY_REF['M6']*0.7 else "⚠️"
print(f"  {status} M6 NN Consistency  OpenBTAI={RESULTS_B['M6_nn_pct']:.1f}%  Cyprus_ref={CY_REF['M6']:.1f}%")

  MORPHOLOGY TESTS M1-M6
  (Cyprus Pure BSF scores shown for reference)
  ⚠️ M1 Volume R²      OpenBTAI=0.104  Cyprus_ref=0.437
  ⚠️ M2 LogVol R²      OpenBTAI=0.126  Cyprus_ref=0.447
  ⚠️ M3 SVR R²         OpenBTAI=-0.188  Cyprus_ref=0.210
  ✅ M4 Necrosis F1    OpenBTAI=0.885  Cyprus_ref=0.695
  ⚠️ M5 Elongation R²  OpenBTAI=-0.009  Cyprus_ref=-0.072
  ✅ M6 NN Consistency  OpenBTAI=21.4%  Cyprus_ref=23.8%


In [16]:
# ── HETEROGENEITY TESTS H1-H4 ──────────────────────────────────────────────
# Cyprus A4B scores for comparison:
# H1 PCA residual Hybrid=0.189  Pure=0.189
# H2 Heterogeneity Hybrid=-0.352  Pure=-0.497
# H3 Subregion F1 Hybrid=0.619  Pure=0.619
# H4 Texture R²  Hybrid=0.421  Pure=0.285

print("=" * 60)
print("  HETEROGENEITY TESTS H1-H4")
print("=" * 60)

# H1: PCA residual (internal structure richness)
pca10 = PCA(n_components=10)
sc_X  = StandardScaler().fit_transform(X_pure[:, :768])
X_pca = pca10.fit_transform(sc_X)
resid = np.linalg.norm(sc_X - pca10.inverse_transform(X_pca), axis=1)
r, _  = pearsonr(resid, np.var(X_pure[:, :768], axis=1))
RESULTS_B['H1_pca_residual_r'] = float(abs(r))
status = "✅" if abs(r) > CY_REF['H1']*0.7 else "⚠️"
print(f"  {status} H1 PCA Residual   OpenBTAI={abs(r):.3f}  Cyprus_ref={CY_REF['H1']:.3f}")

# H2: Heterogeneity (GLCM entropy — log of label variance)
s = cross_val_score(make_probe(n_dim=X_pure.shape[1]), X_pure, glcm_arr, cv=CV_SCAN, scoring='r2')
RESULTS_B['H2_heterogeneity_R2'] = s.mean()
status = "✅" if s.mean() > -0.2 else "⚠️"
print(f"  {status} H2 Heterogeneity  OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['H2']:.3f}")

# H3: Subregion count classification
if len(np.unique(sub_arr)) >= 2:
    s = cross_val_score(make_probe('classification', n_dim=X_pure.shape[1]), X_pure, sub_arr, cv=CV_SCAN, scoring='f1_weighted')
    RESULTS_B['H3_subregion_F1'] = s.mean()
    status = "✅" if s.mean() > CY_REF['H3']*0.7 else "⚠️"
    print(f"  {status} H3 Subregion F1   OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['H3']:.3f}")
else:
    print(f"  ⚪ H3 Subregion — only {len(np.unique(sub_arr))} class(es): {np.unique(sub_arr)}")

# H4: Texture bundle (volume, log-vol, vol^(-1/3))
Y_b = np.column_stack([vol_arr, np.log1p(vol_arr), np.nan_to_num(vol_arr**(-1/3), posinf=0)])
r2s = []
for i in range(Y_b.shape[1]):
    v = np.isfinite(Y_b[:,i]) & (Y_b[:,i] != 0)
    if v.sum() >= 20:
        r2s.append(cross_val_score(make_probe(n_dim=X_pure.shape[1]), X_pure[v], Y_b[v,i], cv=CV_SCAN, scoring='r2').mean())
RESULTS_B['H4_texture_R2'] = float(np.mean(r2s)) if r2s else 0
status = "✅" if RESULTS_B['H4_texture_R2'] > CY_REF['H4']*0.7 else "⚠️"
print(f"  {status} H4 Texture Bundle  OpenBTAI={RESULTS_B['H4_texture_R2']:.3f}  Cyprus_ref={CY_REF['H4']:.3f}")

  HETEROGENEITY TESTS H1-H4
  ✅ H1 PCA Residual   OpenBTAI=0.172  Cyprus_ref=0.189
  ✅ H2 Heterogeneity  OpenBTAI=-0.152  Cyprus_ref=-0.497
  ✅ H3 Subregion F1   OpenBTAI=0.773  Cyprus_ref=0.619
  ⚠️ H4 Texture Bundle  OpenBTAI=0.000  Cyprus_ref=0.285


In [17]:
# ── TEMPORAL TESTS T1, T3-T6 ───────────────────────────────────────────────
# Cyprus A4B scores for comparison:
# T1 dist↔ΔVol  Pure=0.578
# T3 ΔEmb→ΔVol  Pure=-0.549
# T4 Response AUC  Pure=0.509
# T5 Coherence  Pure=0.942
# T6 Velocity   Pure=0.563

print("=" * 60)
print("  TEMPORAL TESTS T1, T3-T6")
print("=" * 60)

# Build sequential scan pairs
pairs = []
for pid, grp in timelines.groupby('patient_id'):
    grp = grp.sort_values('days_since_baseline')
    vs  = grp.to_dict('records')
    for i in range(len(vs) - 1):
        k1 = f"{vs[i]['patient_id']}__{vs[i]['visit_name']}"
        k2 = f"{vs[i+1]['patient_id']}__{vs[i+1]['visit_name']}"
        if k1 in bsf_embeddings and k2 in bsf_embeddings and k1 in volumes_dict and k2 in volumes_dict:
            days = float(max(vs[i+1]['days_since_baseline'] - vs[i]['days_since_baseline'], 1))
            pairs.append({
                'emb1': bsf_embeddings[k1], 'emb2': bsf_embeddings[k2],
                'vol1': volumes_dict[k1]['wt_volume_mm3'],
                'vol2': volumes_dict[k2]['wt_volume_mm3'],
                'days': days
            })

print(f"  Temporal pairs found: {len(pairs)}")

if len(pairs) >= 10:
    # T1: embedding distance correlates with |ΔVol|
    dists = [np.linalg.norm(p['emb2'] - p['emb1']) for p in pairs]
    dvols = [abs(p['vol2'] - p['vol1']) for p in pairs]
    r, _  = pearsonr(dists, dvols)
    RESULTS_B['T1_dist_dvol_r'] = r
    status = "✅" if r > CY_REF['T1']*0.5 else "⚠️"
    print(f"  {status} T1 dist↔|ΔVol|   OpenBTAI={r:.3f}  Cyprus_ref={CY_REF['T1']:.3f}")

    # T3: ΔEmb → ΔVol regression
    X_d = np.array([p['emb2'] - p['emb1'] for p in pairs])
    y_d = np.array([p['vol2'] - p['vol1'] for p in pairs])
    cv3 = KFold(n_splits=min(5, len(pairs)//5), shuffle=True, random_state=42)
    s   = cross_val_score(make_probe(n_dim=X_d.shape[1]), X_d, y_d, cv=cv3, scoring='r2')
    RESULTS_B['T3_delta_R2'] = s.mean()
    status = "✅" if s.mean() > -0.3 else "⚠️"
    print(f"  {status} T3 ΔEmb→ΔVol     OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['T3']:.3f}")

    # T4: Baseline embedding → response (≥20% volume reduction by last scan)
    be, re = [], []
    for pid, grp in timelines.groupby('patient_id'):
        grp = grp.sort_values('days_since_baseline'); vs = grp.to_dict('records')
        if len(vs) < 2: continue
        k0 = f"{vs[0]['patient_id']}__{vs[0]['visit_name']}"
        kL = f"{vs[-1]['patient_id']}__{vs[-1]['visit_name']}"
        if k0 in bsf_embeddings and k0 in volumes_dict and kL in volumes_dict:
            vb = volumes_dict[k0]['wt_volume_mm3']
            if vb > 0:
                be.append(bsf_embeddings[k0])
                re.append(1 if (volumes_dict[kL]['wt_volume_mm3'] - vb)/vb <= -0.2 else 0)
    if len(be) >= 5 and len(set(re)) >= 2:
        Xr, yr = np.array(be), np.array(re)
        n_folds = max(2, min(5, min(sum(yr), len(yr)-sum(yr))))
        try:
            s = cross_val_score(make_probe('classification', n_samples=len(Xr), n_dim=Xr.shape[1]),
                                Xr, yr, cv=StratifiedKFold(n_folds, shuffle=True, random_state=42), scoring='roc_auc')
            RESULTS_B['T4_response_AUC'] = float(s.mean())
            status = "✅" if s.mean() > CY_REF['T4']*0.7 else "⚠️"
            print(f"  {status} T4 Response AUC   OpenBTAI={s.mean():.3f}  Cyprus_ref={CY_REF['T4']:.3f}  (n={len(be)}, responders={sum(yr)})")
        except Exception as ex:
            print(f"  ⚪ T4 Response — {ex}")
    else:
        print(f"  ⚪ T4 Response — insufficient data (n={len(be)}, responders={sum(re)})")

    # T5: Temporal coherence (cosine sim with consecutive scans > non-consecutive)
    emb_arr = np.array([bsf_embeddings[k] for k in eval_keys])
    emb_sc  = StandardScaler().fit_transform(emb_arr)
    key_to_idx = {k: i for i, k in enumerate(eval_keys)}
    consec_sims, non_sims = [], []
    for p in pairs:
        i1 = key_to_idx.get(f"{p['emb1'].tobytes()}", None)
    # Use keys directly
    for pid, grp in timelines.groupby('patient_id'):
        grp = grp.sort_values('days_since_baseline'); vs = grp.to_dict('records')
        for i in range(len(vs) - 1):
            k1 = f"{vs[i]['patient_id']}__{vs[i]['visit_name']}"
            k2 = f"{vs[i+1]['patient_id']}__{vs[i+1]['visit_name']}"
            if k1 in key_to_idx and k2 in key_to_idx:
                a, b = emb_sc[key_to_idx[k1]], emb_sc[key_to_idx[k2]]
                sim = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))
                consec_sims.append(sim)
    # Random non-consecutive
    rng = np.random.default_rng(42)
    for _ in range(len(consec_sims)):
        i, j = rng.choice(len(emb_sc), 2, replace=False)
        if pat_arr[i] != pat_arr[j]:
            a, b = emb_sc[i], emb_sc[j]
            sim = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))
            non_sims.append(sim)
    if consec_sims and non_sims:
        RESULTS_B['T5_coherence'] = np.mean(consec_sims) / (abs(np.mean(non_sims)) + 1e-8)
        status = "✅" if RESULTS_B['T5_coherence'] > CY_REF['T5']*0.7 else "⚠️"
        print(f"  {status} T5 Coherence      OpenBTAI={RESULTS_B['T5_coherence']:.3f}  Cyprus_ref={CY_REF['T5']:.3f}")

    # T6: Velocity consistency (embedding speed ∝ volume change rate)
    speeds = [np.linalg.norm(p['emb2'] - p['emb1']) / p['days'] for p in pairs]
    vol_rates = [abs(p['vol2'] - p['vol1']) / p['days'] for p in pairs]
    r6, _ = pearsonr(speeds, vol_rates)
    RESULTS_B['T6_velocity_r'] = r6
    status = "✅" if r6 > CY_REF['T6']*0.5 else "⚠️"
    print(f"  {status} T6 Velocity       OpenBTAI={r6:.3f}  Cyprus_ref={CY_REF['T6']:.3f}")
else:
    print("  ⚠️  Not enough temporal pairs for temporal tests.")

  TEMPORAL TESTS T1, T3-T6
  Temporal pairs found: 179
  ⚠️ T1 dist↔|ΔVol|   OpenBTAI=0.223  Cyprus_ref=0.578
  ⚠️ T3 ΔEmb→ΔVol     OpenBTAI=-0.778  Cyprus_ref=-0.549
  ⚠️ T4 Response AUC   OpenBTAI=0.300  Cyprus_ref=0.509  (n=32, responders=7)
  ✅ T5 Coherence      OpenBTAI=14.846  Cyprus_ref=0.942
  ✅ T6 Velocity       OpenBTAI=0.300  Cyprus_ref=0.563


In [18]:
# ── SUMMARY TABLE ──────────────────────────────────────────────────────────
print()
print("=" * 65)
print("  OPENBTAI ZERO-SHOT VALIDATION SUMMARY")
print("  (N=253 scans with radiomics coverage)")
print("=" * 65)
print(f"{'Test':<22} {'OpenBTAI':>12} {'Cyprus Pure':>12} {'Status':>8}")
print("-" * 65)

CY_PURE = {'M1':0.437,'M2':0.447,'M3':0.210,'M4':0.695,'M5':-0.072,'M6':23.8,
            'H1':0.189,'H2':-0.497,'H3':0.619,'H4':0.285,
            'T1':0.578,'T3':-0.549,'T4':0.509,'T5':0.942,'T6':0.563}

def row(name, key, ref_key, thresh=0.7):
    v = RESULTS_B.get(key, None)
    rk = CY_PURE.get(ref_key, float('nan'))
    if v is None:
        print(f"  {name:<20}  {'N/A':>10}  {rk:>12.3f}  {'⚪ N/A':>8}")
    else:
        ok = v > rk * thresh if rk >= 0 else v > rk * (2 - thresh)
        print(f"  {name:<20}  {v:>10.3f}  {rk:>12.3f}  {'✅ OK' if ok else '⚠️ LOW':>8}")

row("M1 Volume R²",      'M1_volume_R2',      'M1')
row("M2 LogVol R²",      'M2_logvol_R2',      'M2')
row("M3 SVR R²",         'M3_svr_R2',         'M3')
row("M4 NCR F1",         'M4_necrosis_F1',    'M4')
row("M5 Elongation R²",  'M5_elongation_R2',  'M5', 0.5)
row("M6 NN Consist.",    'M6_nn_pct',         'M6')
row("H1 PCA Residual",   'H1_pca_residual_r', 'H1')
row("H2 Heterogeneity",  'H2_heterogeneity_R2','H2', 0.5)
row("H3 Subregion F1",   'H3_subregion_F1',   'H3')
row("H4 Texture R²",     'H4_texture_R2',     'H4')
row("T1 dist↔ΔVol",     'T1_dist_dvol_r',    'T1', 0.5)
row("T3 ΔEmb→ΔVol",     'T3_delta_R2',       'T3', 0.5)
row("T4 Response AUC",   'T4_response_AUC',   'T4', 0.5)
row("T5 Coherence",      'T5_coherence',      'T5', 0.5)
row("T6 Velocity",       'T6_velocity_r',     'T6', 0.5)

n_ok = sum(1 for k, v in RESULTS_B.items() if v is not None and not np.isnan(v))
print("-" * 65)
print(f"  Tests computed: {n_ok}/15  |  Domain: OpenBTAI (single-modality T1c)")
print("=" * 65)


  OPENBTAI ZERO-SHOT VALIDATION SUMMARY
  (N=253 scans with radiomics coverage)
Test                       OpenBTAI  Cyprus Pure   Status
-----------------------------------------------------------------
  M1 Volume R²               0.104         0.437    ⚠️ LOW
  M2 LogVol R²               0.126         0.447    ⚠️ LOW
  M3 SVR R²                 -0.188         0.210    ⚠️ LOW
  M4 NCR F1                  0.885         0.695      ✅ OK
  M5 Elongation R²          -0.009        -0.072      ✅ OK
  M6 NN Consist.            21.423        23.800      ✅ OK
  H1 PCA Residual            0.172         0.189      ✅ OK
  H2 Heterogeneity          -0.152        -0.497      ✅ OK
  H3 Subregion F1            0.773         0.619      ✅ OK
  H4 Texture R²              0.000         0.285    ⚠️ LOW
  T1 dist↔ΔVol               0.223         0.578    ⚠️ LOW
  T3 ΔEmb→ΔVol              -0.778        -0.549      ✅ OK
  T4 Response AUC            0.300         0.509      ✅ OK
  T5 Coherence              